In [ ]:
import hashlib,json,os,subprocess,sys,tarfile
from pathlib import Path
os.chdir('/kaggle/working')
files=list(Path('/kaggle/input').rglob('payload.bin'))
assert len(files)==1
p=files[0]
assert p.stat().st_size==68315706 and hashlib.sha256(p.read_bytes()).hexdigest()=='8c60ca4670b092354ad88c45a8bcd59c0a8e95f5922c4acf4c4954f2c2dcf936'
root=Path('/kaggle/working/input');root.mkdir(exist_ok=False)
with tarfile.open(p,'r:gz') as tar:
    members=tar.getmembers()
    assert len(members)<2000 and sum(m.size for m in members)<150*1024**2
    assert all(m.isfile() and not Path(m.name).is_absolute() and '..' not in Path(m.name).parts for m in members)
    tar.extractall(root,filter='data')
manifest=root/'INPUT_MANIFEST.json'
assert hashlib.sha256(manifest.read_bytes()).hexdigest()=='d406101f86dfb787a03e7b640789f127264f879cb6fed0d73c62c8b878cf2037'
for item in json.loads(manifest.read_text()):
    path=root/item['name']
    assert path.stat().st_size==item['bytes'] and hashlib.sha256(path.read_bytes()).hexdigest()==item['sha256']
deps=root/'deps'
if deps.exists():
    abi='cp'+str(sys.version_info.major)+str(sys.version_info.minor)
    matches=list(deps.glob('*-'+abi+'-'+abi+'-*.whl'));assert len(matches)==1
    site=Path('/kaggle/working/ortsite')
    subprocess.run([sys.executable,'-m','pip','install','--no-index','--no-deps','--target',str(site),str(matches[0])],check=True)
    sys.path.insert(0,str(site));os.environ['PYTHONPATH']=str(site)
import numpy as np,onnxruntime as ort,onnx,torch
probe=onnx.helper.make_model(onnx.helper.make_graph([onnx.helper.make_node('Identity',['x'],['y'])],'ort_probe',
    [onnx.helper.make_tensor_value_info('x',onnx.TensorProto.FLOAT,[1])],[onnx.helper.make_tensor_value_info('y',onnx.TensorProto.FLOAT,[1])]),opset_imports=[onnx.helper.make_opsetid('',13)])
probe.ir_version=8
assert ort.InferenceSession(probe.SerializeToString(),providers=['CPUExecutionProvider']).run(None,{'x':np.array([1.],np.float32)})[0][0]==1
assert torch.cuda.is_available() and torch.ones(1,device='cuda').item()==1
Path('/kaggle/working/ENVIRONMENT.json').write_text(json.dumps({'python':sys.version,'torch':torch.__version__,'onnxruntime':ort.__version__,'gpu':torch.cuda.get_device_name(0),'actual_ort_operator_probe':'PASS'}))
subprocess.run([sys.executable,str(root/'cloud_worker.py'),'--root',str(root),'--run-id','k230-camera-xy-multiroute-pilot-20261117-a02','--input-sha','d406101f86dfb787a03e7b640789f127264f879cb6fed0d73c62c8b878cf2037','--wall-seconds','2350'],check=True)
